# Hasil — semua angka dibaca dari file

**Riset Deteksi Fault 4 Sensor Kelembaban**

Notebook ini **tidak menghitung ulang model**. Ia membaca file hasil yang
diekspor notebook penelitian, lalu memeriksa ulang angkanya dengan `assert`.
Kalau file berubah, notebook ini langsung gagal — jadi tidak ada angka yang
diketik manual.

Urutan: performa utama → sensor mana yang rusak → catatan data → batasan →
sumber. Semua hasil memakai klasifier **ANN-LM asli**; tidak ada pelatih lain
di edisi ini.

In [ ]:
import os, io, glob, warnings
import numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import display, Image, Markdown

warnings.filterwarnings("ignore")
pd.set_option("display.width", 160)

ASSET_BASE = "https://raw.githubusercontent.com/vousmeevoyez/public-files/main/sidang/assets"
LOCAL_DIRS = ["assets", "sidang/assets", "../sidang/assets"]

def asset_path(rel):
    for base in LOCAL_DIRS:
        p = os.path.join(base, rel)
        if os.path.exists(p):
            return p
    return None

def read_asset(rel):
    p = asset_path(rel)
    if p:
        return pd.read_csv(p)
    r = requests_get(f"{ASSET_BASE}/{rel}")
    return pd.read_csv(io.BytesIO(r))

def requests_get(url):
    import requests
    r = requests.get(url, timeout=120)
    r.raise_for_status()
    return r.content

def show_png(rel, width=900):
    p = asset_path(rel)
    if p:
        display(Image(filename=p, width=width))
    else:
        display(Image(data=requests_get(f"{ASSET_BASE}/{rel}"), width=width))

print("Aset siap dibaca dari:", "folder lokal" if asset_path("11/11_rq4_performa_ann.csv") else "GitHub")

## 1 — Performa utama: EDM-Fuzzy vs JSD-Fuzzy

15 sel = 5 skenario × 3 ukuran window (200, 700, 1000 sampel). Selisih
dihitung JSD − EDM; positif berarti JSD lebih tinggi.

In [ ]:
rq4 = read_asset("11/11_rq4_performa_ann.csv")
p = rq4.pivot_table(index=["N_window", "Scenario"], columns="Method", values="F1").reset_index()
p["selisih"] = p["JSD-Fuzzy"] - p["EDM-Fuzzy"]
p["menang"] = np.where(p["selisih"] > 0, "JSD", "EDM")
p["JSD-Fuzzy"] = p["JSD-Fuzzy"].round(3)
p["EDM-Fuzzy"] = p["EDM-Fuzzy"].round(3)
p["selisih"] = p["selisih"].round(3)

n_win = int((p["selisih"] > 0).sum())
mean_diff = float(p["selisih"].mean())
assert n_win == 11, n_win
assert abs(mean_diff - 0.028) < 0.001, mean_diff

print(f"JSD menang {n_win} dari {len(p)} sel | rata-rata selisih {mean_diff:+.3f}")
display(p.to_string(index=False))

In [ ]:
show_png("11/11_rq4_performa_N700.png", 950)
show_png("11/11_rq4_confusion_matrix_N700.png", 760)

**Cara membaca.**

- JSD menang **11/15**, tetapi tidak merata: N = 700 menang 5/5, sedangkan
  N = 200 dan N = 1000 masing-masing 3/5.
- Selisih terbesar di S1 dengan window panjang (**+0,178**); di S3 beberapa sel
  justru negatif.
- Artinya keunggulan JSD **bergantung kondisi**, bukan otomatis di semua kasus.
  Kalimat yang aman: "konsisten lebih tinggi secara rata-rata".

## 2 — Sensor mana yang rusak?

Setelah model menebak ada fault, rantai kedua menebak sensor mana. Angka di
bawah memakai protokol notebook `11` (ANN-LM asli).

In [ ]:
sr = read_asset("11/11_sensor_ringkas.csv")
q = sr.pivot_table(index=["N_window", "Scenario"], columns="Method", values="F1").reset_index()
q["selisih"] = q["JSD-Fuzzy"] - q["EDM-Fuzzy"]

m_edm = float(sr[sr.Method == "EDM-Fuzzy"].F1.mean())
m_jsd = float(sr[sr.Method == "JSD-Fuzzy"].F1.mean())
n_win_s = int((q["selisih"] > 0).sum())
assert abs(m_jsd - 0.675) < 0.002, m_jsd
assert abs(m_edm - 0.634) < 0.002, m_edm
assert n_win_s == 9, n_win_s

print(f"F1 rata-rata: JSD {m_jsd:.3f} vs EDM {m_edm:.3f} | JSD menang {n_win_s}/15")
per_sensor = (sr[sr.Method == "JSD-Fuzzy"]
              .groupby("Scenario").F1.mean().round(3)
              .rename("F1 JSD per skenario").reset_index())
display(per_sensor.to_string(index=False))
show_png("11/11_sensor_f1.png", 900)

**Cara membaca.** Angka di atas adalah **per skenario**, bukan per sensor
fisik: S1 normal vs fault (0,543), S2 fault tunggal (0,622), S3 kombinasi dua
fault (0,635), S4 kombinasi tiga fault (0,735), S5 kombinasi empat fault (0,841).

Angka pendamping: tebakan **semua empat sensor benar sekaligus** 0,307 (JSD)
vs 0,295 (EDM). Ini lebih ketat daripada F1 rata-rata.

## 3 — Catatan data: sensor 4 pernah macet di 0

Tidak ada baris yang dibuang. Yang perlu dijelaskan saat sidang: kolom
**kelembaban4** bernilai 0 pada **11.371 sampel 30 detik (4,0 %)**, semuanya
antara 16 dan 21 Desember 2025.

| Periode | Panjang |
|---|---|
| 16 Des 07.37–08.01 | 49 sampel (0,4 jam) |
| 17 Des 10.04 – 19 Des 07.03 | 5.399 sampel (45,0 jam) |
| 19 Des 07.57 – 20 Des 21.10 | 4.468 sampel (37,2 jam) |
| 20 Des 22.23 – 21 Des 10.30 | 1.454 sampel (12,1 jam) |

Setelah agregasi 5 menit: 1.135 titik nol. Nilai 0 tetap dipakai karena
**di dalam rentang fisik 0–100 %** dan justru bisa menjadi gejala fault
(pembacaan macet). Dampaknya: **10 dari 280 window N = 200 (3,6 %)** mayoritas
nol; N = 700: 2/79; N = 1000: 1/55. Window itu tetap masuk kelas normal karena
data asli tidak punya label. Ini dicatat sebagai batasan, bukan dibuang.

## 4 — Batasan yang harus disebut

1. Fault berasal dari **simulasi** dengan rumus tetap, bukan kerusakan lapangan.
2. Keputusan butuh window 16,7 jam → pemantauan harian, **bukan alarm real-time**.
3. Keunggulan JSD tidak merata; di S3 beberapa sel negatif.
4. EDM lebih **stabil** (CV lebih rendah di 56/63 pasangan), walau F1 lebih
   rendah. Ini dilaporkan apa adanya.
5. Drift 1 poin/hari sengaja kecil; deteksi drift jangka pendek masih lemah.
6. Sensor 4 macet di 0 selama 5 hari terakhir (11.371 sampel); tidak dibuang,
   tetapi 10/280 window mayoritas nol.

## 5 — Sumber setiap blok

| Blok | Notebook asli | Sel | File |
|---|---|---|---|
| 1 Performa | `11` §10 | 33–40 | `11_rq4_performa_ann.csv` |
| 2 Sensor-ID | `11` §12 | 43–46 | `11_sensor_ringkas.csv` |
| 3 Catatan data | data mentah | — | `data_sensor.csv` |

---
**Status:** privat · commit `27737ee` · 2026-10-26 · rujukan: `sidang/MAPPING.md`